In [ ]:
!pip install "unsloth[kaggle] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install bitsandbytes accelerate -q
!pip install flask flask-ngrok pyngrok protobuf -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from flask import Flask, request, jsonify
from pyngrok import ngrok
import threading

NGROK_TOKEN = ""  
ngrok.set_auth_token(NGROK_TOKEN)

print("Loading Model on GPU...")
base_model_name = "unsloth/Meta-Llama-3.1-8B"
adapter_model_name = "Abdelrahman04/egyptian-educational-chatbot-model"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

print("Loading Adapter...")
model = PeftModel.from_pretrained(base_model, adapter_model_name)
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

app = Flask(__name__)

@app.route("/chat", methods=["POST"])
def chat():
    data = request.json
    student_message = data.get("message", "")
    subject = data.get("subject", "عام")
    history = data.get("history", [])

    print(f"Received: {student_message}")

    prompt = f"موضوع الدرس: {subject}\n\n"
    for user_msg, bot_msg in history:
        prompt += f"الطالب: {user_msg}\nالمعلم: {bot_msg}\n"
    prompt += f"الطالب: {student_message}\nالمعلم:"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.2,
        do_sample=True
    )
    
    full = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = full.split("المعلم:")[-1].strip()
    
    return jsonify({"response": response})

@app.route("/")
def home():
    return "Egyptian Chatbot Server is Running!"

public_url = ngrok.connect(5000).public_url
print(f"🔥 Server Running! Copy this URL to your local GUI: {public_url}")
app.run(port=5000)

2025-12-25 18:18:53.032616: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766686733.059332     315 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766686733.067019     315 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766686733.097394     315 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766686733.097415     315 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766686733.097418     315 computation_placer.cc:177] computation placer alr

Loading Model on GPU...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading Adapter...
🔥 Server Running! Copy this URL to your local GUI: https://adc8773bf6ab.ngrok-free.app
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit


...

127.0.0.1 - - [25/Dec/2025 18:29:15] "POST /chat HTTP/1.1" 200 -


Received: ايه هو قانون الجاذبية؟


127.0.0.1 - - [25/Dec/2025 18:48:43] "POST /chat HTTP/1.1" 200 -


Received: تعريف الكتلة ايه؟


127.0.0.1 - - [25/Dec/2025 18:50:14] "POST /chat HTTP/1.1" 200 -
